In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.nn import functional as F

import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from PIL import Image
import tqdm
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 基本配置
BASE_DIR = Path('/home/thelia/Learn/ML/MyCode/LearningMachineLearning/Thelia/ProjectSekai')
TRAINING_DATA_DIR = BASE_DIR / 'TrainingData'
CODE_DIR = BASE_DIR / 'Codes'

# 超参数
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 5
RANDOM_SEED = 42

# 类别定义
CLASS_NAMES = ['Enanan', 'Kanade', 'Mafuyu', 'Mizuki']
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}

print(f"配置信息:")
print(f"  训练数据目录: {TRAINING_DATA_DIR}")
print(f"  类别数: {NUM_CLASSES}")
print(f"  类别: {CLASS_NAMES}")
print(f"  批大小: {BATCH_SIZE}")
print(f"  学习率: {LEARNING_RATE}")
print(f"  最大epoch数: {NUM_EPOCHS}")

In [ ]:
# ==================== Phase 0: 数据增强 ====================
# 将原始WebP图片转为PNG，并生成4个版本：原图、水平翻、竖直翻、双翻

def flip_and_convert_images(input_dir, output_dir, class_name):
    """
    对指定类别的所有图片进行翻转和格式转换
    
    Args:
        input_dir: 原始WebP图片所在目录
        output_dir: 输出PNG图片目录
        class_name: 类别名称
    
    Returns:
        处理的总图片数量
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    image_count = 0
    webp_files = sorted(input_dir.glob('*.webp'))
    
    print(f"\n处理 {class_name} ({len(webp_files)} 张原始图片)...")
    
    for webp_file in tqdm(webp_files, desc=f"{class_name}"):
        try:
            # 打开原始WebP图片
            img = Image.open(webp_file).convert('RGB')
            
            # 生成4个版本
            base_name = webp_file.stem  # 不含扩展名的文件名
            
            # 1. 原始图片
            img.save(output_dir / f'{base_name}_original.png', 'PNG')
            
            # 2. 水平翻转
            img_h_flip = img.transpose(Image.FLIP_LEFT_RIGHT)
            img_h_flip.save(output_dir / f'{base_name}_h_flip.png', 'PNG')
            
            # 3. 竖直翻转
            img_v_flip = img.transpose(Image.FLIP_TOP_BOTTOM)
            img_v_flip.save(output_dir / f'{base_name}_v_flip.png', 'PNG')
            
            # 4. 双向翻转（等同于180度旋转）
            img_both_flip = img.transpose(Image.FLIP_LEFT_RIGHT).transpose(Image.FLIP_TOP_BOTTOM)
            img_both_flip.save(output_dir / f'{base_name}_both_flip.png', 'PNG')
            
            image_count += 4  # 一张原图生成4张
            
        except Exception as e:
            print(f"  错误处理 {webp_file.name}: {e}")
    
    return image_count

# 执行数据增强
print("=" * 60)
print("开始数据增强：WebP → PNG + 翻转")
print("=" * 60)

# 创建输出目录
augmented_training_data_dir = TRAINING_DATA_DIR
total_images_generated = 0

for class_name in CLASS_NAMES:
    input_class_dir = TRAINING_DATA_DIR / class_name
    output_class_dir = augmented_training_data_dir / class_name
    
    if input_class_dir.exists():
        # 先清空输出目录（如果已存在）
        if output_class_dir.exists():
            import shutil
            shutil.rmtree(output_class_dir)
        
        count = flip_and_convert_images(input_class_dir, output_class_dir, class_name)
        total_images_generated += count
        print(f"✓ {class_name}: 生成 {count} 张图片")
    else:
        print(f"⚠ {class_name} 目录不存在: {input_class_dir}")

print("\n" + "=" * 60)
print(f"数据增强完成！")
print(f"总计生成: {total_images_generated} 张PNG图片")
print("=" * 60)

# 验证：统计每个类别的图片数量
print("\n数据集统计:")
for class_name in CLASS_NAMES:
    class_dir = augmented_training_data_dir / class_name
    if class_dir.exists():
        png_count = len(list(class_dir.glob('*.png')))
        print(f"  {class_name}: {png_count} 张PNG图片")


In [ ]:
# 调试：检查数据目录结构
print("调试：检查目录结构")
print(f"TRAINING_DATA_DIR: {TRAINING_DATA_DIR}")
print(f"存在: {TRAINING_DATA_DIR.exists()}")

if TRAINING_DATA_DIR.exists():
    print(f"\nTRAINING_DATA_DIR 内容:")
    for item in TRAINING_DATA_DIR.iterdir():
        print(f"  {item.name} ({'目录' if item.is_dir() else '文件'})")
        if item.is_dir():
            contents = list(item.glob('*'))
            print(f"    包含 {len(contents)} 项")
            if len(contents) > 0:
                print(f"    样例: {contents[0].name}")
else:
    print("TRAINING_DATA_DIR 不存在！")
